In [1]:
#-------------------------------------------------------------------------------
# Setup and Imports
#-------------------------------------------------------------------------------
'''Installation of Ollama needed'''

import os
import glob
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import lightning as pl
import transformers
import ollama

from tqdm import tqdm
from pytorch_lightning import seed_everything


#-------------------------------------------------------------------------------
# Environment / Device Setup
#-------------------------------------------------------------------------------

def is_notebook():
    return 'JPY_PARENT_PID' in os.environ

USE_CUDA = torch.cuda.is_available()
device = torch.device('cuda' if USE_CUDA else 'cpu')
current_directory = os.getcwd()


#-------------------------------------------------------------------------------
# Reproducibility
#-------------------------------------------------------------------------------

seed_everything(42, workers=True)


#-------------------------------------------------------------------------------
# Data Loading: Value Profiles
#-------------------------------------------------------------------------------

value_questions = pd.read_csv('data/ValueQuestions.csv', low_memory=False)
original_answers = pd.read_csv('data/Dyads_FullData.csv', low_memory=False)

questions = value_questions['Questions'][1:35].tolist()
no_questions = len(questions)


#-------------------------------------------------------------------------------
# Scenario Assignment Across Dyads
#-------------------------------------------------------------------------------

no_dyads = 15
scenario_list = [[] for _ in range(no_dyads)]

scenario_list[0] = ['scenario_2_4', 'scenario_3_5', 'scenario_6_8']
scenario_list[1] = ['scenario_3_5', 'scenario_5_7', 'scenario_7_24']
scenario_list[2] = ['scenario_2_4', 'scenario_5_7', 'scenario_8_36']
scenario_list[3] = ['scenario_2_4', 'scenario_4_31', 'scenario_7_24']
scenario_list[4] = ['scenario_2_4', 'scenario_4_31', 'scenario_7_24']
scenario_list[5] = ['scenario_2_4', 'scenario_3_5', 'scenario_7_24']
scenario_list[6] = ['scenario_2_4', 'scenario_3_5', 'scenario_7_24']
scenario_list[7] = ['scenario_3_5', 'scenario_6_8', 'scenario_8_36']
scenario_list[8] = ['scenario_3_5', 'scenario_6_8', 'scenario_8_36']
scenario_list[9] = ['scenario_1_2', 'scenario_3_5', 'scenario_8_36']
scenario_list[10] = ['scenario_2_4', 'scenario_4_31', 'scenario_6_8']
scenario_list[11] = ['scenario_1_2', 'scenario_3_5', 'scenario_4_31']
scenario_list[12] = ['scenario_1_2', 'scenario_4_31', 'scenario_7_24']
scenario_list[13] = ['scenario_6_8', 'scenario_3_5', 'scenario_2_4']
scenario_list[14] = ['scenario_2_4', 'scenario_5_7', 'scenario_7_24']

print('Number of Dyads:', no_dyads)

In [1]:
#---------------------------------------------------------------------
# Answers and Scenarios
#---------------------------------------------------------------------

answers = '''
Answer 1 = Do not perform CPR if it does not improve a chance at full or partial recovery.
Answer 2 = Do not use a ventilator if it does not allow a chance at full or partial recovery.
Answer 3 = Kept comfortable and pain free while nature takes its course.
Answer 4 = Everything to be done even if it looks like there is no chance of full or partial recovery.
Answer 5 = Proxy to make this decision on his or her own.
'''

scenarios = {
    'scenario_1_2': '''Age: 43
Background: You enjoy an active lifestyle. You like marathons, hiking, and rock climbing. You have been married for 20 years and have two children (16 years and 12 years). You have occasional seasonal allergies, but otherwise, you have been in good health.
Living Situation: You live in a two-story home with a large backyard with a garden. Together with your spouse, you take care of the children, managing their daily activities, schoolwork, and extracurricular activities.
Event: On one of your rock-climbing adventures at the gym, you fell from a significant height while using a safety harness. The angle and force of the fall caused you to hit your head and you lost consciousness. You were transported to the hospital and diagnosed with a traumatic brain injury. You were unable to communicate for some time but later recovered.
Later, a lump in your upper left thigh was found and biopsied.
Investigations: The biopsy revealed a high-grade soft tissue sarcoma with spread to nearby structures.
Clinical Decision: The recommended treatment is hemipelvectomy and left leg amputation at the hip.''',

    'scenario_2_4': '''Age: 20
Background: You are a university student studying literature. You are an introvert and struggle with severe depression, but you have sought counseling through university mental health services. You are an only child.
Living Situation: You live alone in a university dormitory and have a few close friends.
Event: Emergency services responded to a dorm fire. You were found with extensive burn injuries, initially suspected to be self-inflicted.
Investigations: You have third-degree burns over ~80% of your body and were intubated due to airway risk.
Clinical Decision: You require intensive burn care, repeated surgeries, grafting, and long-term rehabilitation.''',

    'scenario_3_5': '''Age: 70
Background: You are a retired librarian with two adult children and four grandchildren. You previously underwent vascular surgery.
You previously expressed preference for comfort-focused care over aggressive intervention.
Living Situation: You live alone in a small apartment.
Event: You developed severe respiratory distress and were diagnosed with COVID-19 pneumonia requiring ventilation.
Clinical Decision: ECMO is being considered, but resources are limited.''',

    'scenario_4_31': '''Age: 35
Background: You are an architect with a successful career and strong community involvement.
Event: You were involved in a structural collapse at a construction site and sustained severe trauma.
Investigations: You have severe TBI, rib fractures, punctured lung, and pelvic injury.
Clinical Decision: You are intubated and in a medically induced coma with high risk of long-term impairment.''',

    'scenario_5_7': '''Age: 77
Background: You are a retired accountant with heart disease and diabetes.
Living Situation: You live alone with support from adult children.
Event: You were admitted with severe heart failure and found to have a lung mass.
Clinical Decision: Biopsy and surgery carry high risk given your condition.''',

    'scenario_6_8': '''Age: 69
Background: You are a retired teacher with COPD.
Event: You were admitted after a fall and developed swallowing difficulty.
Clinical Decision: PEG tube placement is considered but conservative management is an alternative.''',

    'scenario_7_24': '''Age: 68
Background: You are a former coal miner with severe COPD on home oxygen.
Event: You developed respiratory failure requiring ventilation.
Clinical Decision: Tracheostomy is being considered for long-term ventilation support.''',

    'scenario_8_36': '''Age: 78
Background: You are a retired professor with advanced Parkinson’s disease.
Event: You were admitted after a fall complicated by infection and sepsis.
Clinical Decision: You require vasopressors and antibiotics with uncertain prognosis.''' 
}

In [3]:
#------------------------------------------------------------------------
# Prompting the LLM across Dyads and Scenarios
#------------------------------------------------------------------------

subject_scenario1 = [5,3,3,3,5,3,2,3,3,3,1,4,3,2,4]  # Answers picked by patient for scenario 1
subject_scenario2 = [3,3,4,3,3,3,4,4,3,3,1,3,3,3,3]  # Answers picked by patient for scenario 2
subject_scenario3 = [5,3,4,4,3,3,3,3,3,3,3,3,3,3,4]  # Answers picked by patient for scenario 3
proxy_avg_results = [2,1,2,2,1,3,0,0,0,2,0,3,2,0,1]  # Number of answers proxy matched patient

temperature_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]
no_temps = len(temperature_values)
no_trials = 10
columns = ['Dyad', 'Temperature', 'Trial', 'AI Subject Without Values', 'AI Subject With Values','AI Proxy Without Values', 'AI Proxy With Values', 'Human Proxy']

# Load existing data from CSV
try:
    results_df = pd.read_csv('data/LLM_Results.csv')
    processed_dyads = results_df['Dyad'].unique()
    processed_dyads = set(processed_dyads)
except FileNotFoundError:
    results_df = pd.DataFrame(columns=columns)
    processed_dyads = set()

start_dyad = 0 
start_temp = 0.0
start_trial = 0
skip_dyads = True
skip_temps = True
skip_trials = True

## FUNCTION TO EXTRACT THE FIRST NUMBER FROM THE TEXT
def extract_first_number(text):

    match = re.search(r'\d+', text)
    if match: return match.group()
    return None

## LOOPING OVER ALL COMBINATIONS OF PARAMETERS
for dyad in range(no_dyads):

    if skip_dyads:
        if dyad < start_dyad: 
            continue
        else: 
            skip_dyads = False

    print(f'############################### DYAD {dyad+1} #####################################')

    ### OBTAINING VALUES FOR SUBJECT
    values_subject = original_answers.iloc[dyad][2:36]
    updated_values_subject = [f"{question} Answer = {value}. " for question, value in zip(questions, values_subject)]
    updated_values_subject = ''.join(updated_values_subject)

    ### OBTAINING ANSWERS FOR SUBJECT
    subject_answers = [subject_scenario1[dyad], subject_scenario2[dyad], subject_scenario3[dyad]]  

    for temp in range(no_temps):

        temperature = temperature_values[temp]
        if skip_temps:
            if dyad == start_dyad and temperature < start_temp:
                continue
            else:
                skip_temps = False
        print(f'Temperature is {temperature_values[temp]}')

        for trial in range(no_trials):
            if skip_trials:
                if dyad == start_dyad and temperature == start_temp and trial < start_trial:
                    continue
                else:
                    skip_trials = False
            print(f'************* Trial {trial +1} ****************')  

            SUBJECT_NOVALS = []
            SUBJECT_VALS = []
            PROXY_NOVALS = []
            PROXY_VALS = []

            for index, scenario_code in enumerate(scenario_list[dyad]):      

                ### PART 1: SUBJECT ANSWERING FOR THEMSELVES        

                prompt_text1 = '''

                ### TASK: Imagine that you are the patient in this scenario: {SCENARIO_TEXT}. 
                A major clinical decision needs to be made for your treatment in this scenario, and there are five possible choices that you can choose from: {ANSWERS}.
                Imagining that you are the patient, choose only one of the answers that you think would be most appropriate to be applied in this situation using the information in the scenario. 
                Give your answer as a single number corresponding to one of the five options.'''        
                scenario_text = scenarios[scenario_code]
                prompt1 = prompt_text1.format(SCENARIO_TEXT=scenario_text, ANSWERS=answers)
                response1 = ollama.chat(model='llama3', options={'temperature': temperature_values[temp]}, messages=[{'role': 'system', 'content': prompt1}])  
                SUBJECT_NOVALS.append(extract_first_number(response1['message']['content']))

                prompt_text2 = '''         

                ### TASK: Imagine that you are the patient in this scenario: {SCENARIO_TEXT}. 
                The patient (you) were first asked to fill out two questionnaires about your values and wishes in life. These are your answers: {VALUES}. 
                A major clinical decision needs to be made for your treatment in this scenario, and there are five possible choices that you can choose from: {ANSWERS}.
                Imagining that you are the patient, choose only one of the answers that you think would be most appropriate to be applied in this situation using the information in the scenario and the questions from the values questionnaires.        
                Give your answer as a single number corresponding to one of the five options.'''        
                scenario_text = scenarios[scenario_code]
                prompt2 = prompt_text2.format(SCENARIO_TEXT=scenario_text, ANSWERS=answers, VALUES = updated_values_subject)
                response2 = ollama.chat(model='llama3', options={'temperature': temperature_values[temp]}, messages=[{'role': 'system', 'content': prompt2}])  
                SUBJECT_VALS.append(extract_first_number(response2['message']['content']))

                ### PART 2: PROXY ANSWERING FOR SUBJECT   

                prompt_text3 = '''

                ### TASK: Imagine that you are the decision-making proxy (e.g. family member or friend) of the patient in this scenario: {SCENARIO_TEXT}. 
                A major clinical decision needs to be made by you for the patient's treatment in this scenario, and there are five possible choices that you can choose from: {ANSWERS}.
                Imagining that you are the proxy, choose only one of the answers that you think would be most appropriate to be applied in this situation using the information in the scenario. 
                Give your answer as a single number corresponding to one of the five options.'''          
                scenario_text = scenarios[scenario_code]
                prompt3 = prompt_text3.format(SCENARIO_TEXT=scenario_text, ANSWERS=answers)
                response3 = ollama.chat(model='llama3', options={'temperature': temperature_values[temp]}, messages=[{'role': 'system', 'content': prompt3}])
                PROXY_NOVALS.append(extract_first_number(response3['message']['content']))

                prompt_text4 = '''         

                ### TASK: Imagine that you are the decision-making proxy (e.g. family member or friend) of the patient in this scenario: {SCENARIO_TEXT}. 
                The patient was first asked to fill out two questionnaires about their values and wishes in life. These are their answers: {VALUES}. 
                A major clinical decision needs to be made for your treatment in this scenario, and there are five possible choices that you can choose from: {ANSWERS}.
                Imagining that you are the proxy, choose only one of the answers that you think would be most appropriate to be applied in this situation using the information in the scenario and the questions from the values questionnaires. 
                Give your answer as a single number corresponding to one of the five options.'''   
                scenario_text = scenarios[scenario_code]
                prompt4 = prompt_text4.format(SCENARIO_TEXT=scenario_text, ANSWERS=answers, VALUES = updated_values_subject)
                response4 = ollama.chat(model='llama3', options={'temperature': temperature_values[temp]}, messages=[{'role': 'system', 'content': prompt4}])
                PROXY_VALS.append(extract_first_number(response4['message']['content']))

            def count_exact_matches(subject_answers, patient_answers):

                count = 0
                for subj_ans, pat_ans in zip(subject_answers, patient_answers):
                    if str(subj_ans) == pat_ans: count += 1
                return count

            SUBJECT_NOVALS_CORRECT = count_exact_matches(subject_answers, SUBJECT_NOVALS)
            SUBJECT_VALS_CORRECT = count_exact_matches(subject_answers, SUBJECT_VALS)
            PROXY_NOVALS_CORRECT = count_exact_matches(subject_answers, PROXY_NOVALS)
            PROXY_VALS_CORRECT = count_exact_matches(subject_answers, PROXY_VALS)

            new_row = pd.DataFrame([{

                'Dyad': dyad + 1,
                'Temperature': temperature_values[temp],
                'Trial': trial + 1,
                'AI Subject Without Values': SUBJECT_NOVALS_CORRECT,
                'AI Subject With Values': SUBJECT_VALS_CORRECT,
                'AI Proxy Without Values': PROXY_NOVALS_CORRECT,
                'AI Proxy With Values': PROXY_VALS_CORRECT,
                'Human Proxy': proxy_avg_results[dyad]

            }])

            results_df = pd.concat([results_df, new_row], ignore_index=True)

            results_df.to_csv('data/LLM_Results.csv', index=False)


############################### DYAD 1 #####################################
Temperature is 0.0
************* Trial 1 ****************


/scratch/local/5365552/ipykernel_3422903/3820246275.py:140: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, new_row], ignore_index=True)


************* Trial 2 ****************
************* Trial 3 ****************
************* Trial 4 ****************
************* Trial 5 ****************
************* Trial 6 ****************
************* Trial 7 ****************
************* Trial 8 ****************
************* Trial 9 ****************
************* Trial 10 ****************
Temperature is 0.2
************* Trial 1 ****************
************* Trial 2 ****************
************* Trial 3 ****************
************* Trial 4 ****************
************* Trial 5 ****************
************* Trial 6 ****************
************* Trial 7 ****************
************* Trial 8 ****************
************* Trial 9 ****************
************* Trial 10 ****************
Temperature is 0.4
************* Trial 1 ****************
************* Trial 2 ****************
************* Trial 3 ****************
************* Trial 4 ****************
************* Trial 5 ****************
************* Trial 6 **